# E3-10C — LSTM et TCN : prévision multihorizons

**Extension indépendante du notebook historique `08B — LSTM + TCN`.** Horizons **15, 30, 60, 120 min**, entrées séquentielles **5 × 3**, sélection des architectures sur **validation uniquement**, puis entraînement final avec **trois graines (42, 123, 2026)**. L'évaluation principale utilise **13 643 clés de test communes** aux horizons et au protocole original.

Le notebook **ne modifie aucun résultat historique**. Il reproduit les architectures, la grille, l'optimiseur et l'arrêt anticipé de `08B`. La sélection est faite séparément pour chaque horizon, avec la graine 42. Les résultats de test ne sont calculés qu'après la sélection.

**Pré-requis :** exécuter et valider `E3_06`, `E3_07ABCD`, `E3_08` et l'audit corrigé `E3_09`. Installer PyTorch dans l'environnement Jupyter si nécessaire.


In [ ]:
from pathlib import Path
import json, random, time, platform, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path.cwd().resolve()
REP = ROOT / "outputs" / "E3_aligned_representations"
COMMON = ROOT / "outputs" / "E3_common_evaluation"
OUT = ROOT / "outputs" / "E3_10C_LSTM_TCN_multihorizon"
HORIZONS = [15, 30, 60, 120]
SPLITS = ["train", "validation", "test"]
SEEDS = [42, 123, 2026]
KEYS = ["uuid", "timestamp", "split"]

# Identical to original notebook 08B
BATCH_SIZE = 512
MAX_EPOCHS = 60
PATIENCE = 8
LR = 1e-3
WEIGHT_DECAY = 1e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert REP.is_dir() and COMMON.is_dir(), "Execute E3_07ABCD and E3_08 first."
common = pd.read_csv(COMMON / "01_common_keys_all_horizons.csv")
common["timestamp"] = pd.to_datetime(common["timestamp"], utc=True)
assert len(common) == 91763 and not common.duplicated(KEYS).any()
assert common.groupby("split").size().to_dict() == {
    "train": 64393, "validation": 13727, "test": 13643
}
OUT.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("Output:", OUT)
print("Device:", device, "| torch:", torch.__version__)


In [ ]:
def load_horizon(h):
    folder = REP / f"h{h:03d}"
    data = {}
    for split in SPLITS:
        meta = pd.read_csv(folder / f"metadata_{split}.csv")
        meta["timestamp"] = pd.to_datetime(meta["timestamp"], utc=True)
        meta["target_timestamp"] = pd.to_datetime(meta["target_timestamp"], utc=True)
        X = np.load(folder / f"X_sequence_{split}_scaled.npy").astype(np.float32)
        y = np.load(folder / f"y_sequence_{split}.npy").astype(np.float32).reshape(-1)
        X_unscaled = np.load(folder / f"X_sequence_{split}_unscaled.npy", mmap_mode="r")
        idx = np.load(COMMON / f"indices_h{h:03d}_{split}.npy").astype(np.int64)
        assert X.shape == (len(y), 5, 3)
        assert len(meta) == len(y) == len(X_unscaled)
        assert len(idx) == {"train":64393,"validation":13727,"test":13643}[split]
        assert len(np.unique(idx)) == len(idx) and idx.min() >= 0 and idx.max() < len(y)
        assert not meta.duplicated(KEYS).any() and meta["split"].eq(split).all()
        assert (meta["target_timestamp"]-meta["timestamp"]).eq(pd.Timedelta(minutes=h)).all()
        assert np.isfinite(X).all() and np.isfinite(y).all()
        assert np.allclose(meta["target"].to_numpy(), y, atol=1e-5, rtol=0)
        selected = meta.iloc[idx][KEYS].sort_values(KEYS).reset_index(drop=True)
        expected = common.loc[common["split"].eq(split), KEYS].sort_values(KEYS).reset_index(drop=True)
        assert selected.equals(expected), f"Common-key mismatch: h={h}, split={split}"
        # Persistence is computed from unscaled last observed temperature, not from scaled X.
        persistence = np.asarray(X_unscaled[:, -1, 0], dtype=np.float64)
        data[split] = dict(X=X, y=y, meta=meta, idx=idx, persistence=persistence)
        print(f"h={h:3d} {split:10s} X={X.shape} common={len(idx)}")
    return data

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

def metrics(y, pred):
    y = np.asarray(y, dtype=np.float64)
    pred = np.asarray(pred, dtype=np.float64)
    err = pred - y
    sst = np.sum((y - y.mean()) ** 2)
    return {
        "n": int(len(y)),
        "RMSE": float(np.sqrt(np.mean(err**2))),
        "MAE": float(np.mean(np.abs(err))),
        "R2": float(1 - np.sum(err**2)/sst) if sst > 0 else float("nan"),
        "Bias": float(err.mean()),
        "Error_SD": float(err.std(ddof=1))
    }


In [ ]:
# Architectures reproduced from historical 08B.
class LSTMRegressor(nn.Module):
    def __init__(self, hidden=32, layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(3, hidden, num_layers=layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        z, _ = self.lstm(x)
        return self.head(z[:, -1, :]).squeeze(-1)

class CausalConv1d(nn.Module):
    def __init__(self, cin, cout, kernel=2, dilation=1):
        super().__init__()
        self.pad = (kernel - 1) * dilation
        self.conv = nn.Conv1d(cin, cout, kernel, dilation=dilation)
    def forward(self, x):
        return self.conv(nn.functional.pad(x, (self.pad, 0)))

class TCNRegressor(nn.Module):
    def __init__(self, channels=32, kernel=2, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(3, channels, kernel, 1), nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(channels, channels, kernel, 2), nn.ReLU(), nn.Dropout(dropout)
        )
        self.head = nn.Linear(channels, 1)
    def forward(self, x):
        z = self.net(x.transpose(1, 2))
        return self.head(z[:, :, -1]).squeeze(-1)

CANDIDATES = {
    "LSTM": {
        "LSTM_h16": lambda: LSTMRegressor(16, 1, 0.0),
        "LSTM_h32": lambda: LSTMRegressor(32, 1, 0.0),
        "LSTM_h64": lambda: LSTMRegressor(64, 1, 0.0),
    },
    "TCN": {
        "TCN_c16": lambda: TCNRegressor(16, 2, 0.1),
        "TCN_c32": lambda: TCNRegressor(32, 2, 0.1),
        "TCN_c64": lambda: TCNRegressor(64, 2, 0.1),
    }
}

def make_loader(data, split, shuffle=False):
    d = data[split]
    return DataLoader(
        TensorDataset(torch.from_numpy(d["X"]), torch.from_numpy(d["y"])),
        batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0
    )

@torch.no_grad()
def predict(model, data, split):
    model.eval()
    chunks = []
    for xb, _ in make_loader(data, split, False):
        chunks.append(model(xb.to(device)).cpu().numpy())
    return np.concatenate(chunks)

def fit_model(ctor, data, seed, checkpoint):
    # Seed BEFORE constructing model (unlike a common reproducibility pitfall).
    seed_all(seed)
    model = ctor().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    best, bad, history = float("inf"), 0, []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_losses = []
        for xb, yb in make_loader(data, "train", True):
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.item()))
        vp = predict(model, data, "validation")
        vy = data["validation"]["y"]
        vrmse = metrics(vy, vp)["RMSE"]
        history.append({"epoch":epoch, "train_MSE":float(np.mean(train_losses)),
                        "validation_RMSE":vrmse})
        if vrmse < best - 1e-7:
            best, bad = vrmse, 0
            torch.save(model.state_dict(), checkpoint)
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    return model, pd.DataFrame(history), best


## Exécution des entraînements

Pour chaque horizon : **6 entraînements de sélection** (3 LSTM + 3 TCN, graine 42), puis **6 entraînements finaux** (2 architectures retenues × 3 graines). Soit **48 entraînements** au total, avec arrêt anticipé. La durée dépend du matériel.

Les prédictions **test** ne sont produites qu'après la sélection des architectures. Chaque ligne de prédiction conserve `uuid`, `timestamp`, `target_timestamp`, `horizon_min`, `model` et `seed` pour les comparaisons appariées ultérieures.


In [ ]:
selection_rows = []
final_rows = []
prediction_frames = []
device_rows = []
selected_configs = []

for h in HORIZONS:
    print(f"\\n{'='*65}\\nHORIZON {h} MINUTES\\n{'='*65}")
    data = load_horizon(h)
    chosen = {}

    # Phase 1: validation-only selection. No test predictions are computed.
    for family, configs in CANDIDATES.items():
        candidates_for_family = []
        for name, ctor in configs.items():
            path = OUT / f"h{h:03d}_selection_{name}_seed42.pt"
            start = time.perf_counter()
            model, hist, best = fit_model(ctor, data, 42, path)
            hist.to_csv(OUT / f"h{h:03d}_selection_history_{name}.csv", index=False)
            row = {
                "horizon_min":h, "model":family, "configuration":name,
                "selection_seed":42, "validation_RMSE":best,
                "epochs_run":len(hist), "selection_seconds":time.perf_counter()-start
            }
            selection_rows.append(row)
            candidates_for_family.append(row)
            print("Selection:", row)
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        chosen[family] = min(
            candidates_for_family,
            key=lambda r: (r["validation_RMSE"], r["configuration"])
        )["configuration"]
    print("Selected configurations:", chosen)

    # Phase 2: final independent trainings, with selected architecture frozen.
    for family in ("LSTM", "TCN"):
        config = chosen[family]
        for seed in SEEDS:
            start = time.perf_counter()
            checkpoint = OUT / f"h{h:03d}_{family}_seed{seed}.pt"
            model, hist, best = fit_model(CANDIDATES[family][config], data, seed, checkpoint)
            hist.to_csv(OUT / f"h{h:03d}_{family}_seed{seed}_history.csv", index=False)
            elapsed = time.perf_counter() - start
            selected_configs.append({
                "horizon_min":h, "model":family, "configuration":config,
                "seed":seed, "train_n":len(data["train"]["y"]),
                "validation_n":len(data["validation"]["y"]),
                "test_n":len(data["test"]["y"]),
                "best_validation_RMSE_during_training":best,
                "epochs_run":len(hist), "training_seconds":elapsed,
            })

            for split in SPLITS:
                d = data[split]
                y = d["y"]
                p = predict(model, data, split)
                ref = d["persistence"]
                # Descriptive metrics on complete horizon-specific split.
                for scope, indices in [
                    ("full_horizon_split", np.arange(len(y))),
                    ("common_keys", d["idx"])
                ]:
                    if scope == "common_keys" and split != "test":
                        continue
                    m = metrics(y[indices], p[indices])
                    mref = metrics(y[indices], ref[indices])
                    final_rows.append({
                        "horizon_min":h, "model":family, "configuration":config,
                        "seed":seed, "split":split, "scope":scope, **m,
                        "Persistence_RMSE":mref["RMSE"],
                        "skill_RMSE":1-m["RMSE"]/mref["RMSE"]
                    })

                if split == "test":
                    idx = d["idx"]
                    part = d["meta"].iloc[idx].copy().reset_index(drop=True)
                    part["horizon_min"] = h
                    part["model"] = family
                    part["configuration"] = config
                    part["seed"] = seed
                    part["y_true"] = y[idx]
                    part["y_pred"] = p[idx]
                    part["persistence_pred"] = ref[idx]
                    part["error"] = p[idx] - y[idx]
                    prediction_frames.append(part)
                    for uuid, group in part.groupby("uuid", sort=True):
                        yd = group["y_true"].to_numpy()
                        pd_ = group["y_pred"].to_numpy()
                        rd = group["persistence_pred"].to_numpy()
                        mm, rr = metrics(yd, pd_), metrics(yd, rd)
                        device_rows.append({
                            "horizon_min":h, "model":family, "configuration":config,
                            "seed":seed, "uuid":uuid, **mm,
                            "Persistence_RMSE":rr["RMSE"],
                            "skill_RMSE":1-mm["RMSE"]/rr["RMSE"]
                        })
            print(f"Final: h={h}, {family}, seed={seed}, "
                  f"best val RMSE={best:.5f}, {elapsed:.1f}s")
            del model
            if torch.cuda.is_available(): torch.cuda.empty_cache()

# Save outputs ONLY after successful completion of all horizons.
selection_df = pd.DataFrame(selection_rows)
metrics_df = pd.DataFrame(final_rows)
pred_df = pd.concat(prediction_frames, ignore_index=True)
device_df = pd.DataFrame(device_rows)
configs_df = pd.DataFrame(selected_configs)

assert len(selection_df) == 4*2*3
assert len(configs_df) == 4*2*3
assert len(pred_df) == 4*2*3*13643
assert pred_df.groupby(["horizon_min","model","seed"]).size().eq(13643).all()
assert not pred_df.duplicated(["horizon_min","model","seed","uuid","timestamp"]).any()
assert metrics_df.query("split == 'test' and scope == 'common_keys'").shape[0] == 24

selection_df.to_csv(OUT / "01_validation_architecture_selection.csv", index=False)
metrics_df.to_csv(OUT / "02_metrics_all_seeds_all_splits.csv", index=False)
pred_df.to_csv(OUT / "03_common_test_predictions_all_seeds.csv", index=False)
device_df.to_csv(OUT / "04_common_test_per_device_per_seed.csv", index=False)
configs_df.to_csv(OUT / "05_selected_configs_and_training.csv", index=False)

primary = metrics_df.query("split == 'test' and scope == 'common_keys'").copy()
primary.to_csv(OUT / "00_PRIMARY_common_TEST_per_seed.csv", index=False)

summary = (
    primary.groupby(["horizon_min","model"], as_index=False)
    .agg(
        n_seeds=("seed","nunique"), n=("n","first"),
        RMSE_mean=("RMSE","mean"), RMSE_SD=("RMSE","std"),
        MAE_mean=("MAE","mean"), MAE_SD=("MAE","std"),
        R2_mean=("R2","mean"), R2_SD=("R2","std"),
        Bias_mean=("Bias","mean"), Bias_SD=("Bias","std"),
        skill_RMSE_mean=("skill_RMSE","mean"),
        skill_RMSE_SD=("skill_RMSE","std")
    )
)
assert summary["n_seeds"].eq(3).all() and summary["n"].eq(13643).all()
summary.to_csv(OUT / "00_PRIMARY_common_TEST_mean_sd.csv", index=False)

manifest = {
    "source": "E3_aligned_representations",
    "common_keys": "E3_common_evaluation",
    "horizons_minutes": HORIZONS,
    "representation": "sequence [5,3], temperature/RH/CO2",
    "architecture_candidates": {family:list(configs) for family,configs in CANDIDATES.items()},
    "selection": "validation RMSE only; selection seed 42, separately per horizon",
    "final_seeds": SEEDS,
    "training": "full horizon-specific train",
    "test": "13643 common node samples per horizon, per model, per seed",
    "historical_results": "unchanged",
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "optimizer": "Adam",
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "device": str(device),
    "python": platform.python_version(),
    "torch": torch.__version__,
}
(OUT / "06_protocol.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("\\nPRIMARY RESULTS (mean ± SD across training seeds)")
print(summary.to_string(index=False))
print("\\nSaved outputs:", OUT)


## Interprétation et contrôles

- **Comparer d'abord les résultats sur `common_keys`**. Les résultats `full_horizon_split` sont descriptifs et ne doivent pas être mélangés avec le benchmark commun.
- **Ne pas utiliser le test pour sélectionner une architecture** : la sélection utilise la validation et la graine 42, puis chaque architecture sélectionnée est réentraînée avec les trois graines.
- La moyenne ± écart-type sur trois graines mesure une **variabilité d'initialisation**, et non un intervalle de confiance temporel. Une analyse appariée par blocs temporels devra être réalisée séparément.
- Les modèles profonds reçoivent le même historique de **5 × 3** que dans `08B`. Les résultats historiques à 15 minutes sont conservés.
- Le test commun porte sur les mêmes **13 643 clés** aux quatre horizons, mais sur des cibles différentes.
- **Fichiers à transmettre :** `00_PRIMARY_common_TEST_per_seed.csv`, `00_PRIMARY_common_TEST_mean_sd.csv`, `01_validation_architecture_selection.csv`, `04_common_test_per_device_per_seed.csv` et `06_protocol.json`.
